In [ ]:
!pip install -q transformers torch accelerate

import os
import pandas as pd
import torch
from transformers import pipeline
from google.colab import drive

In [ ]:
#Import drive
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:

# Fail immediately if no GPU is available
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not available. In Colab go to Runtime > Change runtime type > GPU, "
        "then reconnect and run again."
    )

device = 0  # first CUDA GPU
print("Using GPU:", torch.cuda.get_device_name(0))

Using GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# Import Model
#Pick an explicit model for reproducibility
#The project used 3 different models.
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
#Set up Sentiment Analysis pipeline

sentiment_classifier = pipeline(
    "sentiment-analysis",
    model=model_name,
    device=device
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
# Your three input files
input_files = [

    "/content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_main.csv",
    "/content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_chunks.csv",
    "/content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_sentences.csv",
]


In [ ]:
# Columns to keep if they exist
preferred_columns = ["doc_id", "text", "iso_code", "year", "final_label"]

for file_path in input_files:
    print(f"\nProcessing: {file_path}")

    df = pd.read_csv(file_path)

    if "text" not in df.columns:
        print(f"Skipped: no 'text' column in {file_path}")
        continue



Processing: /content/gdrive/MyDrive/Colab Notebooks/neo_geo_classfied_main.csv

Processing: /content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_chunks.csv

Processing: /content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_sentences.csv


In [ ]:
# Clean text column
texts = df["text"].fillna("").astype(str).tolist()

In [ ]:
# Run sentiment analysis
results = sentiment_classifier(
        texts,
        batch_size=32,
        truncation=True
    )

In [ ]:
# Keep original columns that actually exist
cols_to_keep = [col for col in preferred_columns if col in df.columns]
results_df = df[cols_to_keep].copy()

In [ ]:
# Add predictions
results_df["sentiment_label"] = [r["label"] for r in results]
results_df["confidence_score"] = [r["score"] for r in results]

In [ ]:
# Save output beside original file
base_name = os.path.splitext(os.path.basename(file_path))[0]
output_path = f"/content/gdrive/MyDrive/Colab Notebooks/{base_name}_sentiment_results.csv"

results_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /content/gdrive/MyDrive/Colab Notebooks/neo_geo_class_sentences_sentiment_results.csv


In [ ]:
# Quick summary
print(results_df["sentiment_label"].value_counts(dropna=False))
display(results_df.head())

sentiment_label
POSITIVE    12708
NEGATIVE     8492
Name: count, dtype: int64


,doc_id,text,iso_code,year,final_label,sentiment_label,confidence_score
0,CHL-1946.14,We do not sell our exportable products at suff...,CHL,1946,neo-liberal,NEGATIVE,0.999561
1,EGY-1946.18,When we consider some of the economic problems...,EGY,1946,geo-economic,NEGATIVE,0.969219
2,GBR-1946.27,The first is to declare again the basic princi...,GBR,1946,neo-liberal,POSITIVE,0.990102
3,GBR-1946.28,"Ever since the Atlantic Charter, every Member ...",GBR,1946,neo-liberal,POSITIVE,0.998854
4,PHL-1946.12,This is not the sort of economic interdependen...,PHL,1946,neo-liberal,NEGATIVE,0.999510
